# Session 2 — Field Maps, Profiles, and Publication-Quality Figures

**Post-CFD Analysis with Python | Dr. Nuha Aljuneidi**

Once data is verified (Session 1), the next job is communication. A well-made contour map or profile plot can make a result obvious; a poorly made one can hide or misrepresent it. This session builds field maps and profile extractions to publication standard.

## Learning outcomes
- Reshape scattered or structured CFD field data for contour plotting.
- Produce labeled, correctly-scaled velocity and pressure contour maps.
- Extract and plot 1D profiles from 2D field data.
- Apply publication-quality figure conventions (units, labels, resolution).

## Using your own Fluent or CSV data
This notebook uses synthetic data so you can run every cell immediately without a CFD license. When you are ready to use your own results, export a CSV from Fluent (or any solver) with coordinates, variable names, units, operating conditions, and a case identifier, then replace the synthetic-data cell below with:

```python
df = pd.read_csv("your_export.csv")
```

Map your solver's column names to the ones used in this notebook before continuing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
print("Environment ready.")

## 1. Building a synthetic structured field

Real Fluent exports on a structured slice can be reshaped into a 2D grid for contouring. Here we generate velocity magnitude and pressure directly on a grid representing flow past a bump on a wall, so every step below mirrors what you would do after `pivot`-ing a real export.

In [ ]:
nx, ny = 120, 60
x_m = np.linspace(0.0, 2.0, nx)
y_m = np.linspace(0.0, 1.0, ny)
X, Y = np.meshgrid(x_m, y_m)

U_inf_mps = 8.0
rho_kgpm3 = 1.225
p_inf_Pa = 101325.0

bump = 0.35 * np.exp(-((X - 1.0) ** 2) / 0.05) * np.exp(-Y / 0.4)
velocity_x_mps = U_inf_mps * (1.0 + bump)
velocity_y_mps = 0.5 * np.gradient(bump, axis=1) * 10
velocity_mag_mps = np.sqrt(velocity_x_mps ** 2 + velocity_y_mps ** 2)
pressure_Pa = p_inf_Pa + 0.5 * rho_kgpm3 * (U_inf_mps ** 2 - velocity_mag_mps ** 2)

print(f"Grid: {nx} x {ny} points, velocity range "
      f"[{velocity_mag_mps.min():.2f}, {velocity_mag_mps.max():.2f}] m/s")

## 2. Publication-quality contour maps

Every field figure needs: a labeled colorbar with units, axis labels with units, equal or clearly stated aspect ratio, and a title that states what is plotted and under what condition.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cf = ax.contourf(X, Y, velocity_mag_mps, levels=25, cmap="viridis")
cbar = fig.colorbar(cf, ax=ax)
cbar.set_label("Velocity magnitude (m/s)")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal")
ax.set_title(f"Velocity magnitude, U_inf = {U_inf_mps:.1f} m/s")
plt.tight_layout()
plt.savefig("velocity_field.png", dpi=300)
plt.show()
print("Saved velocity_field.png at 300 dpi (publication resolution).")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cf = ax.contourf(X, Y, pressure_Pa - p_inf_Pa, levels=25, cmap="RdBu_r")
cbar = fig.colorbar(cf, ax=ax)
cbar.set_label("Gauge pressure, p - p_inf (Pa)")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal")
ax.set_title("Static pressure field (gauge)")
plt.tight_layout()
plt.show()

### Checkpoint 1 — Read your own figure
Look at the pressure figure. Where is pressure lowest, and does that location match where velocity is highest? Explain the physical relationship in one sentence, citing the equation that connects them.

## 3. Extracting a 1D profile

Engineers usually need a specific cut through the field — a boundary-layer profile, a wake profile, a centerline trace — not just the full 2D map. Extract velocity vs. `y` at a fixed `x` location.

In [ ]:
def extract_profile(x_grid, y_grid, field, x_target_m):
    """Extract field(y) at the grid column nearest x_target_m."""
    col = np.argmin(np.abs(x_grid[0, :] - x_target_m))
    return y_grid[:, col], field[:, col], x_grid[0, col]

y_profile, v_profile, x_actual = extract_profile(X, Y, velocity_x_mps, x_target_m=1.0)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(v_profile, y_profile, marker="o", markersize=3)
ax.set_xlabel("velocity_x (m/s)")
ax.set_ylabel("y (m)")
ax.set_title(f"Streamwise velocity profile at x = {x_actual:.2f} m")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Checkpoint 2 — Compare two profiles
Extract the profile again at `x_target_m=0.5` (upstream of the bump) and plot both profiles on the same axes with a legend. Where do they differ most, and does that match the bump location at `x = 1.0` m?

In [ ]:
# TODO: extract the upstream profile, plot both profiles together with a legend


## 4. Publication-quality checklist

Before a figure goes into a report or paper, verify:
1. Every axis has a label with units.
2. Every color map has a labeled colorbar with units.
3. The title states the quantity and the operating condition (not just "Contour Plot 1").
4. Font sizes are legible at the final print size, not just on screen.
5. The figure is saved at ≥300 dpi in a vector or high-resolution raster format.

In [ ]:
# A before/after comparison: an unlabeled figure vs. a publication-ready one
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].contourf(X, Y, velocity_mag_mps, levels=25)
axes[0].set_title("Before: no labels, no units, no colorbar")

cf = axes[1].contourf(X, Y, velocity_mag_mps, levels=25, cmap="viridis")
fig.colorbar(cf, ax=axes[1], label="Velocity magnitude (m/s)")
axes[1].set_xlabel("x (m)")
axes[1].set_ylabel("y (m)")
axes[1].set_aspect("equal")
axes[1].set_title(f"After: U_inf = {U_inf_mps:.1f} m/s, publication-ready")

plt.tight_layout()
plt.show()

### Checkpoint 3 — Fix a broken figure
The "before" panel above is missing at least four of the five checklist items. List which ones, specifically, and why each matters to a reader who was not in the room when you made the plot.

## Graduate/Advanced Extension
Overlay streamlines on the velocity field using `ax.streamplot(x_m, y_m, velocity_x_mps, velocity_y_mps)`. Discuss one situation where streamlines communicate something a contour map cannot (e.g., recirculation, flow separation), and one situation where they can mislead a reader.

## Exit ticket
In three sentences: name one figure-quality mistake you are now confident you would catch in someone else's CFD report, one profile you would want to extract from your own project's flow field, and why a colorbar without units is a reporting error, not a style choice.

**Next:** Session 3 moves from field maps to surface data — pressure coefficients and integrated aerodynamic loads.